In [1]:
import numpy as np
import numpy as np
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
import matplotlib.pyplot as plt

from qiskit_algorithms.optimizers import L_BFGS_B
optimizer = L_BFGS_B(maxiter=500)

In [2]:
# define parameters

N = 6          # number of qubits
J = 1.0        # coupling strength
reps     = 2
tol      = 0.1 
h_values = np.linspace(0.2, 3, 25)   # define your h values


# Finite-size scaling parameters (exact values for 1D TFIM)
N_values = [4, 6, 8, 10]    # system sizes to sweep
beta_exp = 0.125
nu_exp   = 1.0
maxiter  = 500       # increased from 1000
n_restarts = 3        # number of random restarts per point

In [3]:
def build_tfim_hamiltonian(N: int, J: float, h: float) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N - 1):
        pauli_str = ["I"] * N
        pauli_str[i]     = "Z"
        pauli_str[i + 1] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), -J))
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "X"
        pauli_terms.append(("".join(reversed(pauli_str)), -h))
    return SparsePauliOp.from_list(pauli_terms)


def build_ansatz(N: int, reps: int = 2):
    return efficient_su2(num_qubits=N, reps=reps, entanglement="linear")


def build_magnetisation_op(N: int) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), 1 / N))
    return SparsePauliOp.from_list(pauli_terms)


def exact_ground_state_energy(N: int, J: float, h: float) -> float:
    H_matrix = build_tfim_hamiltonian(N, J, h).to_matrix()
    return float(np.linalg.eigvalsh(H_matrix)[0])


def run_vqe(N: int, J: float, h: float, reps: int = 2,
            seed: int = 42, maxiter: int = 500,
            warm_start_params: np.ndarray = None):
    """
    Returns (best_energy, best_params).
    If warm_start_params provided, starts from there (1 attempt).
    Otherwise uses n_restarts random initialisations near the critical point,
    and 1 restart in clearly ordered/disordered regions.
    """
    hamiltonian = build_tfim_hamiltonian(N, J, h)
    ansatz      = build_ansatz(N, reps)
    estimator   = StatevectorEstimator()

    def cost_fn(params):
        pub    = (ansatz, hamiltonian, params)
        result = estimator.run([pub]).result()
        return float(np.real(result[0].data.evs))

    # Adaptive restarts: more near critical point, fewer elsewhere
    near_critical = abs(h / J - 1.0) < 0.3
    attempts      = n_restarts if (near_critical and warm_start_params is None) else 1

    # If warm start provided, use it as the sole starting point
    if warm_start_params is not None:
        starting_points = [warm_start_params]
    else:
        starting_points = [
            np.random.default_rng(seed + i).uniform(-np.pi, np.pi, ansatz.num_parameters)
            for i in range(attempts)
        ]

    best_energy = np.inf
    best_params = None

    for init_params in starting_points:
        result = minimize(
            cost_fn,
            init_params,
            method="L-BFGS-B",           # gradient-based, fast on statevector
            options={"maxiter": maxiter, "ftol": 1e-9, "gtol": 1e-6}
        )
        if result.fun < best_energy:
            best_energy = result.fun
            best_params = result.x

    return best_energy, best_params


def classify_phase(h: float, J: float, tol: float) -> str:
    ratio = h / J
    if abs(ratio - 1.0) < tol:
        return " CRITICAL "
    elif ratio < 1.0:
        return " ORDERED  "
    else:
        return "DISORDERED"


def sweep_phase_diagram(N: int, J: float, h_values: np.ndarray, reps: int = 2):
    """Returns (vqe_energies, exact_energies, magnetisations)."""
    vqe_energies   = []
    exact_energies = []
    magnetisations = []

    M_op      = build_magnetisation_op(N)
    estimator = StatevectorEstimator()
    prev_params = None    # warm start: carry optimal params between h steps

    for i, h in enumerate(h_values):
        phase = classify_phase(h, J, tol)
        print(f"[{i+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

        E_exact = exact_ground_state_energy(N, J, h)
        exact_energies.append(E_exact)

        ansatz = build_ansatz(N, reps)

        # Pass previous params as warm start
        E_vqe, params = run_vqe(N, J, h, reps=reps,
                                 warm_start_params=prev_params)
        prev_params = params    # carry forward for next iteration
        vqe_energies.append(E_vqe)

        pub    = (ansatz, M_op, params)
        result = estimator.run([pub]).result()
        M      = abs(float(np.real(result[0].data.evs)))
        magnetisations.append(M)

        print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
              f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (np.array(vqe_energies),
            np.array(exact_energies),
            np.array(magnetisations))


def run_finite_size_sweep(N_values, J, h_values, reps=2):
    results = {}
    for N in N_values:
        n_reps = reps if N <= 6 else reps + 1
        print(f"\n{'='*50}")
        print(f"Running sweep for N={N}, reps={n_reps}")
        print(f"{'='*50}")
        vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
            N, J, h_values, reps=n_reps
        )
        results[N] = {
            "vqe_energies"  : vqe_energies,
            "exact_energies": exact_energies,
            "magnetisations": magnetisations
        }
    return results

In [ ]:
results = run_finite_size_sweep(N_values, J, h_values, reps=reps)

colors = ["steelblue", "teal", "coral", "purple"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (N, data), color in zip(results.items(), colors):
    axes[0].plot(h_values / J, data["magnetisations"],
                 "o-", label=f"N={N}", color=color)
axes[0].axvline(x=1.0, color="gray", linestyle=":", label="h/J = 1")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("|⟨M⟩|")
axes[0].set_title("Magnetisation vs h/J for increasing N")
axes[0].legend()

for (N, data), color in zip(results.items(), colors):
    axes[1].plot(h_values / J,
                 np.abs(data["vqe_energies"] - data["exact_energies"]),
                 "o-", label=f"N={N}", color=color)
axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|E_VQE - E_exact|")
axes[1].set_title("VQE error vs system size")
axes[1].legend()

plt.tight_layout()
plt.savefig("tfim_finite_size.png", dpi=150)
plt.show()

# Data collapse
fig, ax = plt.subplots(figsize=(8, 5))
for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    ax.plot(x_scaled, y_scaled, "o-", label=f"N={N}", color=color)
ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax.set_title("Finite-size scaling collapse (β=1/8, ν=1)")
ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
ax.legend()
plt.tight_layout()
plt.savefig("tfim_data_collapse.png", dpi=150)
plt.show()


Running sweep for N=4, reps=2
[1/25]  h/J = 0.200  [ ORDERED  ]  E_exact = -3.0617   E_vqe = -3.0600   err = 0.0018   |M| = 0.9874
[2/25]  h/J = 0.317  [ ORDERED  ]  E_exact = -3.1606   E_vqe = -3.1504   err = 0.0102   |M| = 0.9680
[3/25]  h/J = 0.433  [ ORDERED  ]  E_exact = -3.3137   E_vqe = -3.2821   err = 0.0317   |M| = 0.9378
[4/25]  h/J = 0.550  [ ORDERED  ]  E_exact = -3.5242   E_vqe = -3.4553   err = 0.0689   |M| = 0.8948
[5/25]  h/J = 0.667  [ ORDERED  ]  E_exact = -3.7887   E_vqe = -3.6711   err = 0.1176   |M| = 0.8348
[6/25]  h/J = 0.783  [ ORDERED  ]  E_exact = -4.0983   E_vqe = -4.0980   err = 0.0002   |M| = 0.0000
[7/25]  h/J = 0.900  [ ORDERED  ]  E_exact = -4.4427   E_vqe = -4.4425   err = 0.0001   |M| = 0.0000
[8/25]  h/J = 1.017  [ CRITICAL ]  E_exact = -4.8130   E_vqe = -4.8129   err = 0.0001   |M| = 0.0000
[9/25]  h/J = 1.133  [DISORDERED]  E_exact = -5.2024   E_vqe = -5.2023   err = 0.0001   |M| = 0.0000
[10/25]  h/J = 1.250  [DISORDERED]  E_exact = -5.6060   E_vq